## Dataset 2 — Arxiv post-2024 (multi-sauts)
80 résumés sur les LLMs publiés après le 1er janvier 2024

## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation des dépendances

In [ ]:
!pip install -q wikipedia-api arxiv feedparser requests beautifulsoup4

## 2. Imports et configuration

In [ ]:
import os, json, time, datetime, re
import wikipediaapi, arxiv, feedparser
from tqdm.notebook import tqdm
from collections import Counter

RAW_PATH = os.path.join(BASE_PATH, 'data', 'raw')
os.makedirs(RAW_PATH, exist_ok=True)

WIKI_PATH    = os.path.join(RAW_PATH, 'wikipedia_technique.json')
ARXIV_PATH   = os.path.join(RAW_PATH, 'arxiv.json')
LEMONDE_PATH = os.path.join(RAW_PATH, 'lemonde.json')
print(f"Répertoire de sortie : {RAW_PATH}")

## Dataset 1 — Wikipedia FR (technique)
40 articles sur 9 thèmes clés de l'IA

In [ ]:
# ~70 topics Wikipedia FR pour garantir 40 articles même avec des pages manquantes
WIKI_TOPICS = [
    # Réseaux de neurones
    "Réseau de neurones artificiel", "Réseau de neurones récurrent",
    "Réseau de neurones convolutif", "Perceptron", "Rétropropagation du gradient",
    "Réseau résiduel",
    # Transformeur & attention
    "Transformeur (apprentissage automatique)", "Mécanisme d'attention",
    # Apprentissage automatique
    "Apprentissage automatique", "Apprentissage profond",
    "Apprentissage supervisé", "Apprentissage non supervisé",
    "Apprentissage par transfert", "Surajustement",
    "Forêt d'arbres décisionnels", "Machine à vecteurs de support",
    "Régression logistique", "Descente de gradient stochastique",
    # NLP
    "Traitement automatique des langues", "Traduction automatique",
    "Analyse de sentiments", "Résumé automatique",
    "Reconnaissance d'entités nommées", "Question-réponse automatique",
    "Génération de texte", "Lemmatisation", "Tokenisation",
    # Apprentissage par renforcement
    "Apprentissage par renforcement",
    "Apprentissage par renforcement à partir de retours humains",
    "Q-learning",
    # GAN & Diffusion
    "Réseau antagoniste génératif", "Stable Diffusion", "DALL-E",
    "Modèle de diffusion",
    # Modèles de langage
    "BERT (modèle de langage)", "GPT", "ChatGPT", "GPT-4", "LLaMA",
    "Mistral AI", "Claude (assistant)", "Génération augmentée par récupération",
    "Intelligence artificielle générative",
    # IA générale
    "Intelligence artificielle", "Hallucination (intelligence artificielle)",
    "Ingénierie des prompts", "Biais algorithmique",
    "Éthique de l'intelligence artificielle", "Alignement de l'IA",
    # Vision
    "Vision par ordinateur", "Reconnaissance de formes",
    "Détection d'objets", "Segmentation sémantique",
    # Orgas / infra
    "Hugging Face", "OpenAI", "Anthropic",
    # Concepts généraux
    "Réseau de neurones à convolution", "Mémoire à court terme",
    "Apprentissage fédéré", "Compression de modèle",
    "Base de données vectorielle", "Recherche sémantique",
    "Système de recommandation", "Traitement du signal",
    "Informatique cognitive", "Système expert",
]
TARGET_WIKI = 40
print(f"Topics disponibles : {len(WIKI_TOPICS)}")
print(f"Cible              : {TARGET_WIKI} articles (dataset_type='technique')")

In [ ]:
wiki = wikipediaapi.Wikipedia(
    language='fr',
    user_agent='LLM-Integration-Study/1.0 (research@example.com)'
)
wikipedia_articles = []

for topic in tqdm(WIKI_TOPICS, desc="Wikipedia FR [technique]"):
    if len(wikipedia_articles) >= TARGET_WIKI:
        break
    try:
        page = wiki.page(topic)
        if not page.exists():
            print(f"  [SKIP] Inexistante : {topic}")
            continue
        content = page.text
        if len(content) < 300:
            print(f"  [SKIP] Trop court  : {topic}")
            continue
        wikipedia_articles.append({
            "id":           f"wiki_{len(wikipedia_articles):03d}",
            "title":        page.title,
            "content":      content[:8000],
            "source":       "wikipedia_fr",
            "langue":       "fr",
            "date":         datetime.date.today().isoformat(),
            "url":          page.fullurl,
            "dataset_type": "technique",
        })
        time.sleep(0.3)
    except Exception as e:
        print(f"  [ERROR] {topic} : {e}")

total_words_wiki = sum(len(a['content'].split()) for a in wikipedia_articles)
print(f"\nWikipedia collectés : {len(wikipedia_articles)} / {TARGET_WIKI}  (~{total_words_wiki:,} mots)")

## 4. Scraping Arxiv — 160 résumés stratifiés (3 périodes temporelles)

In [ ]:
from datetime import datetime, timezone
import time as _time
import requests as _requests
from bs4 import BeautifulSoup

UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'

ARXIV_QUERIES = [
    "large language models reasoning evaluation 2024",
    "large language models retrieval augmented generation 2024",
    "large language models fine-tuning instruction following 2024",
    "large language models hallucination factuality 2024",
    "large language models multimodal vision 2024",
    "LLM agent tool use planning 2025",
    "LLM preference optimization RLHF DPO 2024",
    "large language models benchmark evaluation 2025",
    "transformer language model pretraining 2024",
    "LLM safety robustness alignment 2024",
    "in-context learning prompt engineering LLM 2024",
    "LLM code generation software engineering 2024",
]
TARGET_ARXIV = 80
DATE_FROM    = datetime(2024, 1, 1, tzinfo=timezone.utc)
ENRICH_AR5IV = True   # Mettre False pour désactiver si ar5iv.org est lent

def get_arxiv_intro(entry_url, max_chars=2500, verbose=False):
    """Récupère l'introduction d'un papier Arxiv.
    Tente dans l'ordre :
      1. arxiv.org/html/{id}   — rendu HTML officiel (meilleur pour papiers récents)
      2. ar5iv.org/abs/{id}    — version LaTeXML
      3. ar5iv.labs.arxiv.org/html/{id}
    """
    try:
        # Extraction ID : http://arxiv.org/abs/2401.12345v1 -> 2401.12345
        arxiv_id = entry_url.split('/abs/')[-1]
        arxiv_id = re.sub(r'v\d+$', '', arxiv_id).strip()

        urls_to_try = [
            f"https://arxiv.org/html/{arxiv_id}",
            f"https://ar5iv.org/abs/{arxiv_id}",
            f"https://ar5iv.labs.arxiv.org/html/{arxiv_id}",
        ]

        soup = None
        for url in urls_to_try:
            try:
                r = _requests.get(url, headers={'User-Agent': UA},
                                  timeout=12, allow_redirects=True)
                if r.status_code == 200 and len(r.text) > 2000:
                    soup = BeautifulSoup(r.text, 'html.parser')
                    if verbose:
                        print(f"    OK  {url}")
                    break
                elif verbose:
                    print(f"    {r.status_code} {url}")
            except Exception as e:
                if verbose:
                    print(f"    ERR {url}: {e}")

        if soup is None:
            return ''

        # 1. Section Introduction par ID (ar5iv/arxiv HTML)
        intro = None
        for sid in ['S1', 'Sx1', 's1', 'intro', 'introduction']:
            intro = soup.find('section', {'id': sid})
            if intro:
                break

        # 2. Heading contenant 'introduction'
        if not intro:
            for h in soup.find_all(['h1', 'h2', 'h3', 'h4']):
                if 'introduction' in h.get_text().lower():
                    intro = h.find_parent('section') or h.parent
                    break

        # 3. Fallback : premiers paragraphes du corps
        if not intro:
            body = (soup.find('article') or
                    soup.find('main') or
                    soup.find('div', class_=lambda c: c and 'ltx_document' in (c or '')))
            if body:
                paras = [p for p in body.find_all('p')
                         if len(p.get_text().strip()) > 40][:8]
                text = ' '.join(p.get_text().strip() for p in paras)
                text = re.sub(r'\s+', ' ', text).strip()
                return text[:max_chars] if len(text.split()) > 30 else ''
            return ''

        text = intro.get_text()
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars]
    except Exception as e:
        if verbose:
            print(f"    EXCEPTION: {e}")
        return ''
    finally:
        _time.sleep(0.3)

# ── Scraping principal ────────────────────────────────────────────────
arxiv_papers = []
seen_ids     = set()

for query in tqdm(ARXIV_QUERIES, desc="Arxiv [multisauts]"):
    if len(arxiv_papers) >= TARGET_ARXIV:
        break
    try:
        search = arxiv.Search(
            query=query,
            max_results=200,
            sort_by=arxiv.SortCriterion.SubmittedDate,
            sort_order=arxiv.SortOrder.Descending,
        )
        client = arxiv.Client()
        for result in client.results(search):
            if len(arxiv_papers) >= TARGET_ARXIV:
                break
            paper_id = result.entry_id
            if paper_id in seen_ids:
                continue
            pub_date = result.published
            if not pub_date or pub_date < DATE_FROM:
                continue
            seen_ids.add(paper_id)
            arxiv_papers.append({
                "id":           f"arxiv_{len(arxiv_papers):03d}",
                "title":        result.title,
                "content":      result.summary,   # Enrichi après
                "source":       "arxiv",
                "langue":       "en",
                "date":         pub_date.date().isoformat(),
                "url":          result.entry_id,
                "authors":      [a.name for a in result.authors[:5]],
                "dataset_type": "multisauts",
            })
        _time.sleep(1)
    except Exception as e:
        print(f"  [ERROR] '{query[:50]}' : {e}")

print(f"Arxiv collectés (résumés seuls) : {len(arxiv_papers)} / {TARGET_ARXIV}")

# ── Enrichissement ar5iv — résumé + introduction ──────────────────────
if ENRICH_AR5IV and arxiv_papers:
    # Test de diagnostic sur le 1er papier avant de lancer les 80
    print("Test diagnostic ar5iv sur le 1er papier...")
    test_intro = get_arxiv_intro(arxiv_papers[0]['url'], verbose=True)
    if test_intro:
        print(f"  OK — {len(test_intro.split())} mots extraits")
    else:
        print("  [WARN] Introductions non disponibles (ar5iv inaccessible ou structure HTML différente)")
        print("  → Le corpus Arxiv contiendra uniquement les résumés (~150 mots/papier)")
        print("  → Mettre ENRICH_AR5IV = False pour ignorer cette étape")

    print(f"\nEnrichissement ({len(arxiv_papers)} papiers)...")
    enriched = 0
    for i, paper in enumerate(tqdm(arxiv_papers, desc="ar5iv intro")):
        verbose_mode = (i < 3)   # Affiche le détail pour les 3 premiers
        intro = get_arxiv_intro(paper['url'], verbose=verbose_mode)
        if intro:
            paper['content'] = paper['content'] + "\n\nIntroduction:\n" + intro
            enriched += 1
    import statistics as _stats
    words_after = [len(p['content'].split()) for p in arxiv_papers]
    print(f"  Papiers enrichis      : {enriched} / {len(arxiv_papers)}")
    print(f"  Mots/papier (médiane) : {_stats.median(words_after):.0f}")
    print(f"  Mots/papier (moyenne) : {_stats.mean(words_after):.0f}")
else:
    print("  (enrichissement ar5iv désactivé ou aucun papier collecté)")

total_words_arxiv = sum(len(p['content'].split()) for p in arxiv_papers)
dates = sorted(p['date'] for p in arxiv_papers if p['date'])
print(f"\nArxiv total : {len(arxiv_papers)} papiers  ~{total_words_arxiv:,} mots")
if dates:
    print(f"  Période : {dates[0]} → {dates[-1]}")
if len(arxiv_papers) < TARGET_ARXIV:
    print(f"  [WARN] Cible non atteinte : {len(arxiv_papers)}/{TARGET_ARXIV}")

## Dataset 3 — Le Monde RSS (temporel)
60 articles d'actualité technologie & sciences via flux RSS

In [ ]:
from bs4 import BeautifulSoup
import time as _time
import requests as _requests

# ── Constantes ────────────────────────────────────────────────────────
TARGET_LEMONDE = 60
MIN_WORDS      = 150   # Seuil minimal pour qu'un article soit conservé
UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'

# ── Scraper générique ─────────────────────────────────────────────────
def scrape_article(url, selectors=None):
    """Scrape le contenu texte d'un article à partir de son URL.
    selectors = liste de (tag, class_fragment) à essayer dans l'ordre."""
    selectors = selectors or [
        ('p', 'article__paragraph'),   # Le Monde
        ('p', 'a-paragraph'),          # France Info
        ('p', 'article-text'),
    ]
    try:
        resp = _requests.get(url, headers={'User-Agent': UA}, timeout=12)
        if resp.status_code != 200:
            return ''
        soup = BeautifulSoup(resp.text, 'html.parser')
        for tag, cls_frag in selectors:
            paras = soup.find_all(tag, class_=lambda c: c and cls_frag in c)
            if paras:
                text = ' '.join(p.get_text().strip() for p in paras)
                text = re.sub(r'\s+', ' ', text).strip()
                if len(text.split()) >= MIN_WORDS:
                    return text[:6000]
        # Fallback générique : balise <article>
        art = soup.find('article')
        if art:
            text = ' '.join(p.get_text().strip()
                            for p in art.find_all('p') if len(p.get_text().strip()) > 30)
            text = re.sub(r'\s+', ' ', text).strip()
            return text[:6000] if len(text.split()) >= MIN_WORDS else ''
        return ''
    except Exception as e:
        return ''
    finally:
        _time.sleep(0.6)   # Respectueux du serveur

# ── Flux RSS à essayer ────────────────────────────────────────────────
LEMONDE_FEEDS = [
    ("le-monde/une",           "https://www.lemonde.fr/rss/une.xml"),
    ("le-monde/sciences",      "https://www.lemonde.fr/sciences/rss_full.xml"),
    ("le-monde/ia",            "https://www.lemonde.fr/intelligence-artificielle/rss_full.xml"),
    ("le-monde/pixels",        "https://www.lemonde.fr/pixels/rss_full.xml"),
    ("le-monde/economie",      "https://www.lemonde.fr/economie/rss_full.xml"),
    ("le-monde/international", "https://www.lemonde.fr/international/rss_full.xml"),
    ("le-monde/societe",       "https://www.lemonde.fr/societe/rss_full.xml"),
    ("le-monde/planete",       "https://www.lemonde.fr/planete/rss_full.xml"),
    ("le-monde/culture",       "https://www.lemonde.fr/culture/rss_full.xml"),
]

# France Info — fallback 100 % open access
FRANCEINFO_FEEDS = [
    ("france-info/titres",    "https://www.francetvinfo.fr/titres.rss"),
    ("france-info/sciences",  "https://www.francetvinfo.fr/sciences.rss"),
    ("france-info/economie",  "https://www.francetvinfo.fr/economie.rss"),
    ("france-info/societe",   "https://www.francetvinfo.fr/societe.rss"),
    ("france-info/politique", "https://www.francetvinfo.fr/politique.rss"),
    ("france-info/monde",     "https://www.francetvinfo.fr/monde.rss"),
]

# ── Extraction RSS → collecte des articles ────────────────────────────
def collect_from_feeds(feeds, target, existing_urls=None):
    """Parcourt des flux RSS, scrape chaque article, ne garde que >= MIN_WORDS."""
    articles = []
    seen = set(existing_urls or [])
    for feed_name, feed_url in feeds:
        if len(articles) >= target:
            break
        try:
            resp = _requests.get(feed_url, headers={'User-Agent': UA}, timeout=15)
            if resp.status_code != 200:
                print(f"  [SKIP] {feed_name} → HTTP {resp.status_code}")
                continue
            feed = feedparser.parse(resp.content)
            print(f"  Flux '{feed_name}' : {len(feed.entries)} entrées RSS")
            for entry in feed.entries:
                if len(articles) >= target:
                    break
                url = entry.get('link', '')
                if url in seen:
                    continue
                seen.add(url)
                # Scraping de l'article complet
                full_text = scrape_article(url)
                if not full_text:
                    # Fallback : titre + résumé RSS (accepté même si court)
                    rss_summary = entry.get('summary', '')
                    rss_summary = re.sub(r'<[^>]+>', ' ', rss_summary)
                    rss_summary = re.sub(r'\s+', ' ', rss_summary).strip()
                    full_text = (entry.get('title','') + '. ' + rss_summary).strip()
                    if len(full_text.split()) < MIN_WORDS:
                        continue   # Trop court même avec le fallback RSS
                # Date
                pub_date = ''
                if hasattr(entry, 'published_parsed') and entry.published_parsed:
                    try:
                        pub_date = _time.strftime('%Y-%m-%d', entry.published_parsed)
                    except Exception:
                        pass
                if not pub_date:
                    pub_date = str(entry.get('published', ''))[:10]
                source_name = feed_name.split('/')[0]
                articles.append({
                    "id":           f"news_{len(articles):03d}",
                    "title":        entry.get('title', '').strip(),
                    "content":      full_text[:6000],
                    "source":       source_name,
                    "langue":       "fr",
                    "date":         pub_date,
                    "url":          url,
                    "feed":         feed_name,
                    "dataset_type": "temporel",
                })
                words = len(full_text.split())
                print(f"    ✓ {entry.get('title','')[:45]:<45}  {words:>4} mots")
        except Exception as e:
            print(f"  [ERROR] {feed_name} : {e}")
    return articles, seen

# ── Passe 1 : Le Monde ────────────────────────────────────────────────
print("Passe 1 — Le Monde (scraping article complet, filtre >= {MIN_WORDS} mots)...")
lemonde_articles, seen_urls = collect_from_feeds(LEMONDE_FEEDS, TARGET_LEMONDE)
print(f"  Le Monde : {len(lemonde_articles)} / {TARGET_LEMONDE} articles ({MIN_WORDS}+ mots)")

# ── Passe 2 : France Info si quota non atteint ────────────────────────
if len(lemonde_articles) < TARGET_LEMONDE:
    remaining = TARGET_LEMONDE - len(lemonde_articles)
    print(f"\nPasse 2 — France Info (fallback open-access, besoin de {remaining} articles)...")
    fi_articles, _ = collect_from_feeds(FRANCEINFO_FEEDS, remaining, seen_urls)
    # Re-numéroter les IDs
    for i, a in enumerate(fi_articles):
        a['id'] = f"news_{len(lemonde_articles)+i:03d}"
    lemonde_articles.extend(fi_articles)
    print(f"  France Info : {len(fi_articles)} articles ajoutés")

total_words_lemonde = sum(len(a['content'].split()) for a in lemonde_articles)
sources = {}
for a in lemonde_articles:
    sources[a['source']] = sources.get(a['source'], 0) + 1

print(f"\nActualités collectées : {len(lemonde_articles)} / {TARGET_LEMONDE}")
print(f"  Mots total           : {total_words_lemonde:,}  (~{total_words_lemonde/max(1,len(lemonde_articles)):.0f} mots/article)")
print(f"  Sources              : {sources}")
if len(lemonde_articles) < TARGET_LEMONDE:
    print(f"  [WARN] Cible non atteinte : {len(lemonde_articles)}/{TARGET_LEMONDE}")

## 5. Sauvegarde des données brutes sur Drive

In [ ]:
# Sauvegarde des 3 datasets sur Google Drive
for data, path, label in [
    (wikipedia_articles, WIKI_PATH,    'Wikipedia technique'),
    (arxiv_papers,       ARXIV_PATH,   'Arxiv multisauts'),
    (lemonde_articles,   LEMONDE_PATH, 'Le Monde temporel'),
]:
    try:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        words = sum(len(d['content'].split()) for d in data)
        print(f"  {label:<25}: {len(data):>3} docs  ~{words:>7,} mots  {path}")
    except Exception as e:
        print(f"  [ERROR] {label} : {e}")

## 6. Statistiques sur les données brutes

In [ ]:
import statistics

all_docs = wikipedia_articles + arxiv_papers + lemonde_articles
all_words = sum(len(d['content'].split()) for d in all_docs)

print("=" * 60)
print("STATISTIQUES — Données brutes")
print("=" * 60)

for label, docs in [("Wikipedia technique", wikipedia_articles),
                    ("Arxiv multisauts",     arxiv_papers),
                    ("Le Monde temporel",    lemonde_articles)]:
    if not docs:
        continue
    wc = [len(d['content'].split()) for d in docs]
    print(f"\n  {label} ({len(docs)} docs)")
    print(f"    Mots/doc  : min={min(wc)}  moy={statistics.mean(wc):.0f}  médiane={statistics.median(wc):.0f}  max={max(wc)}")
    print(f"    Total     : {sum(wc):,} mots")

print(f"\n{'─'*45}")
print(f"  TOTAL : {len(all_docs)} docs — {all_words:,} mots")
print(f"{'─'*45}")
target_ok = '✔ ATTEINT' if all_words >= 100000 else f'⚠ {all_words:,} mots'
print(f"  Objectif 100k+ mots : {target_ok}")

## 6. Résumé final

In [ ]:
all_docs  = wikipedia_articles + arxiv_papers + lemonde_articles
all_words = sum(len(d['content'].split()) for d in all_docs)

print("=" * 65)
print("RÉSUMÉ — Notebook 01 : 3 datasets collectés")
print("=" * 65)
print(f"\n{'Source':<25} {'Docs':>5} {'~Mots':>8}   dataset_type")
print("-" * 65)
for label, docs, dtype in [
    ("Wikipedia FR",   wikipedia_articles, "technique"),
    ("Arxiv",          arxiv_papers,       "multisauts"),
    ("Le Monde",       lemonde_articles,   "temporel"),
]:
    w = sum(len(d['content'].split()) for d in docs)
    print(f"  {label:<23} {len(docs):>5} {w:>8,}   {dtype}")
print("-" * 65)
print(f"  {'TOTAL':<23} {len(all_docs):>5} {all_words:>8,}")

print("\n  Q&A prévues (5 paires/doc sauf Arxiv: 3 simples + 2 complexes) :")
print(f"    Wikipedia  : {len(wikipedia_articles)*5:>4} paires → 100 train + 40 test")
print(f"    Arxiv      : {len(arxiv_papers)*5:>4} paires → 100 train (50s+50c) + 40 test (20s+20c)")
print(f"    Le Monde   : {len(lemonde_articles)*5:>4} paires → 100 train + 40 test")
print(f"    TOTAL      :  300 train + 120 test")

print("\n✔ Notebook 01 terminé. Lancez 02_dataset_builder.ipynb.")
print("=" * 65)